In [ ]:
from __future__ import (absolute_import, division,
                        print_function, unicode_literals)

import warnings
warnings.simplefilter('ignore')

# general purpose packages
import pandas as pd
import numpy as np
import os
import json
import time
import re
import csv
import subprocess
import sys

import scipy.stats as stats
import statsmodels.stats as smstats
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from dotenv import load_dotenv
from pathlib import Path

from multiprocessing import Process, Manager, Pool
import multiprocessing
from functools import partial

from collections import Counter

import seaborn as sns; sns.set()

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
matplotlib.rcParams['backend'] = "Qt5Agg"
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

from IPython.display import display, Image

from adjustText import adjust_text
import builtins
%matplotlib inline

# for normalization
from sklearn.linear_model import QuantileRegressor

# for survival analysis
import sklearn
from sklearn import set_config

from statsmodels.regression.quantile_regression import QuantReg

# for working with yaml files
import ruamel.yaml

import itertools

In [ ]:
def get_pvalue_star(pval, thr=0.05):
    if thr == 0.05:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.05:
            return "*"
        else:
            return ""
    elif thr == 0.1:
        if pval < 0.001:
            return "***"
        elif pval < 0.01:
            return "**"
        elif pval < 0.1:
            return "*"
        else:
            return ""

In [ ]:
# 1. Load the environment variables
load_dotenv("APA_localization.scicore.env")

# 2. Reconstruct the subdirs dictionary
subdirs = {
    "lab_group_dir": os.getenv("LAB_GROUP_DIR"),
    "raw_sequencing_data_dir": os.getenv("RAW_SEQUENCING_DATA_DIR"),
    "main_project_dir": os.getenv("MAIN_PROJECT_DIR"),
    "wf_dir": os.getenv("WF_DIR"),
    "UCSCtracks_dir": os.getenv("UCSC_TRACKS_DIR"),
    "UCSCtracks_trackfiles_dir": os.getenv("UCSC_TRACKFILES_DIR"),
    "UCSCtracks_trackhubs_dir": os.getenv("UCSC_TRACKHUBS_DIR"),
    "human_annotation_dir": os.getenv("HUMAN_ANNOTATION_DIR"),
    "mouse_annotation_dir": os.getenv("MOUSE_ANNOTATION_DIR"),
    "shared_project_dir": os.getenv("SHARED_PROJECT_DIR"),
    "temp_dir": os.getenv("TEMP_DIR"),
    "slurm_dir": os.getenv("SLURM_DIR"),
    "slurm_scripts_dir": os.getenv("SLURM_SCRIPTS_DIR"),
    "figures_dir": os.getenv("FIGURES_DIR"),
    "tables_dir": os.getenv("TABLES_DIR"),
    "fastq_dir": os.getenv("FASTQ_DIR"),
    "metadata_dir": os.getenv("METADATA_DIR"),
    "wf_runs_dir": os.getenv("WF_RUNS_DIR"),
}

# 3. Reconstruct the file_paths dictionary
file_paths = {
    "human_genome_file": os.getenv("HUMAN_GENOME_FILE"),
    "human_chrom_sizes_file": os.getenv("HUMAN_CHROM_SIZES_FILE"),
    "human_annotation_file": os.getenv("HUMAN_ANNOTATION_FILE"),
    "human_basic_annotation_file": os.getenv("HUMAN_BASIC_ANNOTATION_FILE"),
    "human_polyAsite_atlas": os.getenv("HUMAN_POLYASITE_ATLAS"),
    "human_tandem_PAS": os.getenv("HUMAN_TANDEM_PAS"),
    "human_exonic_segments_gtf": os.getenv("HUMAN_EXONIC_SEGMENTS_GTF"),
    "human_exonic_segments_bed": os.getenv("HUMAN_EXONIC_SEGMENTS_BED"),
}

# 4. Safely create all subdirectories
# Using os.makedirs is highly preferred over os.system('mkdir -p')
# because it avoids opening a subshell and handles permissions gracefully in pure Python.
for path in subdirs.values():
    if path:  # Safety check to ensure the variable was actually found in the .env
        os.makedirs(path, exist_ok=True)

print("Environment loaded and directories verified.")

# Get tandem polyAsites customly, restricted by BASIC Gencode annotation, and define regions for gene expression quantification 

In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

from zavolab_pyutils.annotation import (
    get_terminal_exons,
    parse_gtf_attributes_into_pd_dataframes,
    get_GTF_for_gene_expression_analysis,
)

In [ ]:
gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(file_paths['human_basic_annotation_file'])

# for further analysis, retain only protein-coding genes
exons_df = exons_df.loc[exons_df['gene_type']=='protein_coding'].reset_index(drop=True)
genes_df = genes_df.loc[genes_df['gene_type']=='protein_coding'].reset_index(drop=True)

print(f"retained {len(exons_df)} exons in {len(genes_df)} protein-coding genes")

# get .bed file with Terminal exons
TE_bed_df = get_terminal_exons(
    exons_df.copy()
)

# get gtf to be used as input for FeatureCounts (in the workflow), to analyze gene expression
gtf_for_expr_analysis_df = get_GTF_for_gene_expression_analysis(exons_df,file_paths['human_exonic_segments_gtf'])

# Prepare start samples

In [ ]:
# we extract the paths to fastq files, which were originally downloaded into a different Project directory
fractionation_fastq_dir = subdirs['lab_group_dir']+'RNA_subcellular_localization/fastq/'

os.system("""find """+fractionation_fastq_dir+""" -name '*.fastq.1.gz' > """+subdirs['temp_dir']+"""fractionation_fastq_file_paths_1.tsv""")
os.system("""find """+fractionation_fastq_dir+""" -name '*.fastq.2.gz' > """+subdirs['temp_dir']+"""fractionation_fastq_file_paths_2.tsv""")

In [ ]:
fastq_file_paths_1 = pd.read_csv(subdirs['temp_dir']+'fractionation_fastq_file_paths_1.tsv',delimiter="\t",
                                   index_col=None,header=None)
fastq_file_paths_1.columns = ['fq2'] # read 1 and read 2 were messed up
fastq_file_paths_2 = pd.read_csv(subdirs['temp_dir']+'fractionation_fastq_file_paths_2.tsv',delimiter="\t",
                                   index_col=None,header=None)
fastq_file_paths_2.columns = ['fq1'] # read 1 and read 2 were messed up
fastq_file_paths_1['sample'] = fastq_file_paths_1.apply(lambda x:x['fq2'].split('/')[-1].split('.')[0],1)
fastq_file_paths_2['sample'] = fastq_file_paths_2.apply(lambda x:x['fq1'].split('/')[-1].split('.')[0],1)
fastq_file_paths = pd.merge(fastq_file_paths_1,fastq_file_paths_2,how='inner',on='sample')

# filter out unnecessary files
fastq_file_paths = fastq_file_paths.loc[~fastq_file_paths['sample'].str.contains('-Tg-')].reset_index(drop=True)

# filter out bad files - those were crashing during mapping?
# UPDATE: we've managed to "sanitize" these files by removing broken lines. So, now we don't need to filter out these samples.

# bad_samples = ['dLoRNA-DMSO-3-2','dLoRNA-DMSO-1-4','dLoRNA-DMSO-2-4','dLoRNA-DMSO-2-8','dLoRNA-DMSO-3-8',]
# bad_samples = ['dLoRNA-DMSO-3-8']
bad_samples = []

fastq_file_paths = fastq_file_paths.loc[~fastq_file_paths['sample'].isin(bad_samples)].reset_index(drop=True)
fastq_file_paths['source'] = 'CLUSTER'
fractionation_samples = fastq_file_paths[['sample','source','fq1','fq2']].copy()
fractionation_samples['merged_sample'] = fractionation_samples['sample']

In [ ]:
# only look at subcellular Fractionation samples
cols = ['sample','merged_sample','source','fq1','fq2']
start_samples = pd.concat([
                            fractionation_samples[cols],
                          ]).reset_index(drop=True)

In [ ]:
# write to file
start_samples.to_csv(subdirs['metadata_dir']+'start_samples.U2OS_subcellular_fractions.tsv', sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

# Prepare .yaml config file and run WF

create conda environment with snakemake and install SLURM executor

run from the login node on HPC cluster:

```bash
conda create -c conda-forge -c bioconda -n snakemake snakemake
conda activate snakemake
pip install snakemake-executor-plugin-slurm
```

In [ ]:
organism = 'human'

gtf_chrs = pd.read_csv(file_paths[organism+'_basic_annotation_file'],delimiter="\t",
                                   index_col=None,header=None,usecols = [0],skiprows=5)
chromosome_list = list(gtf_chrs[0].unique())
chromosome_list = [elem for elem in chromosome_list if elem not in ['chrY','chrM']] # exclude chrY and chrM from quantification

In [ ]:
# load default rule_config, modify it and save
WF_version = 'U2OS_subcellular_fractions'

yaml = ruamel.yaml.YAML()
yaml.preserve_quotes = True
with open(subdirs['wf_dir']+'config.yaml') as f_read:
    data = yaml.load(f_read)
data['samples_file'] = subdirs['metadata_dir']+'start_samples.U2OS_subcellular_fractions.tsv'

data['output_dir'] = subdirs['wf_runs_dir']+WF_version+'/output/'
data['local_log'] = subdirs['wf_runs_dir']+WF_version+'/output/local_log/'
data['cluster_log'] = subdirs['wf_runs_dir']+WF_version+'/output/cluster_log/'

data['organism'] = organism
data['genome_file'] = file_paths[organism+'_genome_file']
data['gtf_file'] = file_paths[organism+'_basic_annotation_file']
data['exonic_segments_bed'] = file_paths[organism+'_exonic_segments_bed']
data['exonic_segments_gtf'] = file_paths[organism+'_exonic_segments_gtf']

data['chromosomes'] = ' '.join(chromosome_list)
data['tandem_pas'] = file_paths[organism+'_tandem_PAS']

data['PAQ_min_distance_start_to_proximal'] = "1"
data['PAQ_coverage_unstranded'] = "yes"

data['PAQ_min_length_mean_coverage'] = "50"
data['PAQ_min_mean_exon_coverage'] = "1"

data['PAQ_coverage_downstream_extension'] = "100"
data['PAQ_distal_downstream_extension'] = "50"

data['PAQ_max_mean_coverage'] = "50"
data['PAQ_cluster_distance'] = "1"
data['PAQ_upstream_cluster_extension'] = "50"

data['PAQ_coverage_mse_ratio_limit'] = "0.5"

data['PAQ_fragment_length'] = "1"

for dir_path in [data['output_dir'],data['local_log'],data['cluster_log']]:
    command = 'mkdir -p '+dir_path
    out = subprocess.check_output(command, shell=True)

with open(subdirs['wf_runs_dir']+WF_version+'/config_'+'_and_'.join(chromosome_list)+'.yaml','w') as f_write:     
    yaml.dump(data, f_write)

# the following steps should be executed one after another
# uncomment the step to print the snakemake command below
# NOTE - there is an argument "-np" in the end - that results in a mock run, creating a DAG of jobs, to check the correctness of the planned run.
# To actually run, just delete the "-np" from the end of the command.

WF_step = "prepare-faster"
# WF_step = "quantification-faster"
# WF_step = "PAQR-quantify"

command = """snakemake \
--snakefile """+subdirs['wf_dir']+"""Snakefile-"""+WF_step+""" \
--scheduler greedy \
--configfile """+subdirs['wf_runs_dir']+WF_version+'/config_'+'_and_'.join(chromosome_list)+'.yaml'+""" \
--printshellcmds \
--software-deployment-method conda apptainer \
--conda-frontend conda \
--apptainer-args "--bind """+subdirs['wf_dir']+','+subdirs['lab_group_dir']+','+subdirs['raw_sequencing_data_dir']+"""" \
--executor slurm \
--profile """+subdirs['wf_dir']+'profile'+""" \
--nolock \
-np"""

print(command)

execute the command above from login node

snakemake automatically submits and monitors computational jobs to SLURM queue manager

The results of all workflow steps will be necessary to conduct further data analysis below, e.g. analyzing gene expression and alternative polyadenylation (APA)

# Gene expression vs APA

In [ ]:
organism = 'human'
WF_version = 'v1_'+organism

os.system("""find """+subdirs['wf_runs_dir']+WF_version+'/output/gene_expression_quantification/'+organism+'/FeatureCounts_exonic_segments/'+""" -name '*.exonic_segments.txt' > """+subdirs['temp_dir']+"""exonic_segments_FeatureCounts.files.txt""")
os.system("""find """+subdirs['wf_runs_dir']+WF_version+'/output/gene_expression_quantification/'+organism+'/FeatureCounts_standard/'+""" -name '*.standard.txt' > """+subdirs['temp_dir']+"""standard_FeatureCounts.files.txt""")

In [ ]:
exonic_segments_FeatureCounts_files = pd.read_csv(subdirs['temp_dir']+'exonic_segments_FeatureCounts.files.txt',delimiter="\t",
                                   index_col=None,header=None)
exonic_segments_FeatureCounts_files['sample'] = exonic_segments_FeatureCounts_files.apply(lambda x:x[0].split('/')[-1].replace('.exonic_segments.txt',''),1)

# we use by default specific gene expression quantification for which we've created a custom .gtf file above. If instead want to use standard quantification with GENCODE annotation, uncomment:

# exonic_segments_FeatureCounts_files = pd.read_csv(subdirs['temp_dir']+'standard_FeatureCounts.files.txt',delimiter="\t",
#                                    index_col=None,header=None)
# exonic_segments_FeatureCounts_files['sample'] = exonic_segments_FeatureCounts_files.apply(lambda x:x[0].split('/')[-1].replace('.standard.txt',''),1)

exonic_segments_FeatureCounts_files = exonic_segments_FeatureCounts_files.rename(columns={0:'path'})

start_samples = pd.read_csv(subdirs['metadata_dir']+'start_samples.tsv',delimiter="\t",index_col=None,header=0)

metadata = pd.merge(start_samples[['sample']],exonic_segments_FeatureCounts_files,how='inner',on='sample')

In [ ]:
i=0
for index,row in metadata.iterrows():
    tmp = pd.read_csv(row['path'],delimiter="\t",index_col=None,header=0,skiprows=1)
    cols = list(tmp.columns)
    tmp = tmp.rename(columns={cols[-1]:row['sample']})
    if i==0:
        res = tmp.copy()
    else:
        res = pd.merge(res,tmp,how='outer',on=cols[:-1])
    if i%5==0:
        print(str(i)+' done')
    i=i+1

## Fractionation U2OS

### Gene expression analysis

In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2
# this is important to be able to re-import the module after making modifications to the zavolab_pyutils code on Scicore

In [ ]:
sel_metadata = metadata.loc[(metadata['sample'].str.contains('-DMSO-'))].reset_index(drop=True)
sel_metadata['condition'] = sel_metadata.apply(lambda x:x['sample'].split('-')[0]+('-'+str(x['sample'].split('-')[-1]) if len(x['sample'].split('-'))==4 else ''),1)

condition_interpretation_dict = {
    'dTotal':'whole cell',
    'dLoRNA-1':'ER+mitochondria (1)',
    'dLoRNA-2':'ER+mitochondria (2)',
    'dLoRNA-3':'unclear (3)',
    'dLoRNA-4':'nucleolus (4)',
    'dLoRNA-5':'unclear (5)',
    'dLoRNA-6':'membraneless cytosol (6)',
    'dLoRNA-7':'membraneless cytosol (7)',
    'dLoRNA-8':'membraneless cytosol (8)',
}

sel_metadata['condition'] = sel_metadata['condition'].map(condition_interpretation_dict)
sel_metadata['fraction_number'] = sel_metadata.apply(lambda x:int(x['sample'].split('-')[-1]) if len(x['sample'].split('-'))==4 else 0,1)

sel_metadata['replicate'] = sel_metadata.apply(lambda x:x['sample'].split('-')[2],1).astype('int')

index_cols = ['Geneid', 'Chr', 'Start', 'End', 'Strand', 'Length']

samples_list = list(sel_metadata['sample'])
cur_res = res[index_cols+samples_list].reset_index(drop=True)
cur_res[samples_list] = cur_res[samples_list].fillna(0).astype('int')

#### Deseq style analysis

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization
)

norm_cur_res_df, sfs_df = apply_deseq2_normalization(cur_res,sel_metadata.copy(),pseudocount=1)
norm_cur_res_df = pd.concat([cur_res[index_cols],norm_cur_res_df],axis=1) # add index cols

In [ ]:
from zavolab_pyutils.visualization import (
    plot_size_factors,
)

plot_size_factors(sfs_df,savefig_path=subdirs['figures_dir']+'Fractionation_U2OS/diagnostic_plots/library_size_vs_SF.png')

In [ ]:
# these are the samples in right-bottom area:
sfs_df.loc[((sfs_df['sf']<1.5)&(sfs_df['read_sum_mln']>35))]

In [ ]:
log2_norm_cur_res_df = norm_cur_res_df.copy() # make a log2 version for PCA and may be smth else
log2_norm_cur_res_df[samples_list] = np.log2(log2_norm_cur_res_df[samples_list]) # now we have normalized counts with respect to library size

In [ ]:
from zavolab_pyutils.annotation import (
    parse_gtf_attributes_into_pd_dataframes,
)

gtf_df, genes_df, exons_df = parse_gtf_attributes_into_pd_dataframes(file_paths['human_basic_annotation_file'])

In [ ]:
# Run PCA

from zavolab_pyutils.visualization import (
    pca_plot
)

log2_norm_cur_res_df['m'] = log2_norm_cur_res_df[samples_list].mean(axis=1)
PCA_UMAP_subset = log2_norm_cur_res_df.loc[log2_norm_cur_res_df['m']>log2_norm_cur_res_df['m'].quantile(0.3)].copy().reset_index(drop=True)
print(len(PCA_UMAP_subset))

savefig_path = (
    subdirs['figures_dir']+'Fractionation_U2OS/PCA_gene_expression.Deseq_norm_counts.png'
)

palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']
hue_order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    "condition",
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    s_param=40,
    figsize=(4.2,4.2),
)

#### Sanity style analysis

In [ ]:
from zavolab_pyutils.read_count_data_analysis import (
    apply_deseq2_normalization, get_MultiDimR2,
    prepare_isoform_sanity_matrix, 
    apply_sanity_normalization_full_bayesian, 
    test_differential_relative_usage
)

In [ ]:
# we perform Sanity normalization and variance estimation

raw_counts_df = cur_res.copy()
raw_counts_df.index = raw_counts_df['Geneid'].values

# Sanity-normalized counts are already log2-transformed
sanity_norm_counts_df, sanity_means_df, sanity_relative_errors_df, sanity_absolute_errors_df, sanity_vg_df, median_lib_size, variances_df = apply_sanity_normalization_full_bayesian(
    counts_df=raw_counts_df, 
    metadata_df=sel_metadata.copy(), 
    sample_col='sample', 
    cond_col='condition',
    vmin=0.0000001, 
    vmax=100,
    n_cores=5,
    empirical_bayes=True,
    loess_variance_threshold_q=0.25,
)

In [ ]:
# Diagnostic plots for Sanity outputs

from zavolab_pyutils.visualization import (
    plot_variance_vs_expression,
    plot_mean_vs_cv,
)

# discard the effect from library size, as was done in original Sanity paper
ToCompare_sanity_norm_counts_df = sanity_norm_counts_df-np.log2(median_lib_size)

sanity_vg_df['inferred_v_g'] = sanity_vg_df['MAP_v_g'] # use MAP estimate for Vg to plot, in-line with original Sanity implementation
plot_variance_vs_expression(
    ToCompare_sanity_norm_counts_df, sanity_vg_df,
    savefig_path=subdirs['figures_dir']+'Fractionation_U2OS/pySanity/variance_vs_expr.png',
    true_vg=None, # This is to compare with True value specified during simulation above
    ylim = (0,10),
)

natScale_ToCompare_sanity_norm_counts_df = 2**ToCompare_sanity_norm_counts_df

sanity_plot_data = plot_mean_vs_cv(
natScale_ToCompare_sanity_norm_counts_df, sel_metadata.copy(),
savefig_path=subdirs['figures_dir']+'Fractionation_U2OS/pySanity/cv_vs_mean_plot.png')

sel_conditions = ['whole cell','ER+mitochondria (1)','nucleolus (4)','membraneless cytosol (7)']

sanity_plot_data_sel_conditions = plot_mean_vs_cv(
natScale_ToCompare_sanity_norm_counts_df, sel_metadata.loc[sel_metadata['condition'].isin(sel_conditions)].copy(),
savefig_path=subdirs['figures_dir']+'Fractionation_U2OS/pySanity/cv_vs_mean_plot.selected_conditions.png')

In [ ]:
# Run PCA - now based on Sanity normalized counts

from zavolab_pyutils.visualization import (
    pca_plot
)

sanity_norm_counts_df['m'] = sanity_norm_counts_df[samples_list].mean(axis=1)
PCA_UMAP_subset = sanity_norm_counts_df.sort_values(['m'],ascending=[False]).head(12258).copy().reset_index(drop=True) # to have exactly the same number of genes as above for Deseq2 style normalization
print(len(PCA_UMAP_subset))

savefig_path = (
    subdirs['figures_dir']+'Fractionation_U2OS/PCA_gene_expression.Sanity_norm_counts.png'
)

palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']
hue_order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']

pca_plot(
    PCA_UMAP_subset,
    samples_list,
    sel_metadata,
    "condition",
    savefig_path,
    sns_color_palette = palette,
    hue_order = hue_order,
    calculate_permanova_R2=True,
    s_param=40,
    figsize=(4.2,4.2),
)

In [ ]:
# plot expression of NFYA gene across conditions along with its 95% confidence intervals (multiple testing bonferroni-adjusted)
from zavolab_pyutils.visualization import (
    plot_sanity_gene_expression_with_ci,
)

sel_gene_names = ['NFYA']

selected_genes = list(genes.loc[genes['gene_name'].isin(sel_gene_names)]['gene_id'])

order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']
palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']

plot_sanity_gene_expression_with_ci(
    sample_norm_df=sanity_norm_counts_df, 
    means_df=sanity_means_df, 
    errors_df=sanity_absolute_errors_df, 
    metadata_df=sel_metadata.copy(), 
    selected_genes=selected_genes, 
    adjust_multiple_comparisons=True,
    savefig_path=subdirs['figures_dir']+'Fractionation_U2OS/pySanity/over_genes/'+'_'.join(sel_gene_names)+'.expression.png',
    condition_order = order,
    palette = palette,
)

In [ ]:
# for NFYA
sel_gene_name = 'NFYA'

order_sample_list = list(sel_metadata.sort_values(['fraction_number','replicate'])['sample'])
temp_df = pd.merge(genes[['gene_id','gene_name','gene_type']].rename(columns={'gene_id':'Geneid'}),cur_res,how='right',on=['Geneid'])

sel_gene_expression_df = pd.DataFrame(temp_df.loc[temp_df['gene_name']==sel_gene_name][order_sample_list].transpose())
sel_gene_expression_df.columns = ['gene_expression']

sel_gene_expression_df['sample'] = sel_gene_expression_df.index
sel_gene_expression_df = sel_gene_expression_df.reset_index(drop=True)
sel_gene_expression_df = pd.merge(sel_gene_expression_df,sel_metadata,how='left',on='sample')
data = sel_gene_expression_df.copy()


alpha_param, s_param = 0.9, 10
sns.set(font_scale=1.2)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(3.4,3.2))

y_feature,x_feature='condition','gene_expression'
order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']
palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']

ax = sns.stripplot(data = data,x=x_feature,y=y_feature,s=s_param,alpha=alpha_param,edgecolor='black',linewidth=0.5,
                     order = order, palette = palette,orient='h')
ax.tick_params(bottom=True,left=True)
ax.set(xlabel = sel_gene_name+' expression level, $log_2$',ylabel='')

out = subprocess.check_output('mkdir -p '+subdirs['figures_dir']+'Fractionation_U2OS/over_genes/', shell=True)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+sel_gene_name+'.expression.png',bbox_inches='tight',dpi=600)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+sel_gene_name+'.expression.pdf',bbox_inches='tight',dpi=600)

In [ ]:
# for NFYA
sel_gene_name = 'CD47'

order_sample_list = list(sel_metadata.sort_values(['fraction_number','replicate'])['sample'])
temp_df = pd.merge(genes[['gene_id','gene_name','gene_type']].rename(columns={'gene_id':'Geneid'}),cur_res,how='right',on=['Geneid'])

sel_gene_expression_df = pd.DataFrame(temp_df.loc[temp_df['gene_name']==sel_gene_name][order_sample_list].transpose())
sel_gene_expression_df.columns = ['gene_expression']

sel_gene_expression_df['sample'] = sel_gene_expression_df.index
sel_gene_expression_df = sel_gene_expression_df.reset_index(drop=True)
sel_gene_expression_df = pd.merge(sel_gene_expression_df,sel_metadata,how='left',on='sample')
data = sel_gene_expression_df.copy()


alpha_param, s_param = 0.9, 10
sns.set(font_scale=1.2)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(3.4,3.2))

y_feature,x_feature='condition','gene_expression'
order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']
palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']

ax = sns.stripplot(data = data,x=x_feature,y=y_feature,s=s_param,alpha=alpha_param,edgecolor='black',linewidth=0.5,
                     order = order, palette = palette,orient='h')
ax.tick_params(bottom=True,left=True)
ax.set(xlabel = sel_gene_name+' expression level, $log_2$',ylabel='')

out = subprocess.check_output('mkdir -p '+subdirs['figures_dir']+'Fractionation_U2OS/over_genes/', shell=True)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+sel_gene_name+'.expression.png',bbox_inches='tight',dpi=600)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+sel_gene_name+'.expression.pdf',bbox_inches='tight',dpi=600)

### Shortening of terminal exons via APA

In [ ]:
organism = 'human'
WF_version = 'v1_'+organism

os.system("""find """+subdirs['wf_runs_dir']+WF_version+'/output/PAQR/'+organism+'/quantifications/'+""" -name 'tandem_pas_expression-mean*.tsv' > """+subdirs['temp_dir']+"""tandem_PAS_counts.fractionation_and_HCT.files.txt""")

In [ ]:
samples_list = list(sel_metadata['sample'])

tandem_PAS_counts_files = pd.read_csv(subdirs['temp_dir']+"""tandem_PAS_counts.fractionation_and_HCT.files.txt""",delimiter="\t",
                                   index_col=None,header=None)

i=0
a = []
for index,row in tandem_PAS_counts_files.iterrows():
    tmp = pd.read_csv(row[0],delimiter="\t",index_col=None,header=0)
    cols = list(tmp.columns)
    index_cols = cols[:10]
    sample_cols = cols[10:]
    col_rename_dict = {}
    for elem in sample_cols:
        col_rename_dict[elem] = elem.split('.chr')[0]
    tmp = tmp.rename(columns=col_rename_dict)[index_cols+samples_list]
    a.append(tmp)
    if i%4==0:
        print(str(i)+' done')
    i=i+1
tandem_PAS_rc = pd.concat(a).reset_index(drop=True)
tandem_PAS_rc[samples_list] = tandem_PAS_rc[samples_list].astype('int')

#### Relative usage

In [ ]:
# calculate relative usage

gr = tandem_PAS_rc[['exon']+samples_list].groupby('exon').sum().reset_index()
k=0
for col in samples_list:
    if k==0:
        relative_usages = pd.merge(tandem_PAS_rc,gr[['exon',col]].rename(columns={col:'sum'}),how='left',on='exon')
    else:
        relative_usages = pd.merge(relative_usages,gr[['exon',col]].rename(columns={col:'sum'}),how='left',on='exon')
    relative_usages[col] = relative_usages[col]/relative_usages['sum']
    relative_usages = relative_usages.drop(['sum'],axis=1)
    k=k+1

In [ ]:
organism = 'human'

gtf = pd.read_csv(file_paths[organism+'_basic_annotation_file'],delimiter="\t",index_col=None,header=None,skiprows=5)

genes = gtf.loc[gtf[2]=='gene'].reset_index(drop=True)
genes['gene_type'] = genes[8].str.split('gene_type "',expand=True)[1].str.split('";',expand=True)[0]
genes['gene_name'] = genes[8].str.split('gene_name "',expand=True)[1].str.split('";',expand=True)[0]
genes['gene_id'] = genes[8].str.split('gene_id "',expand=True)[1].str.split('";',expand=True)[0]

In [ ]:
genes.loc[genes['gene_name']=='NFYA']

In [ ]:
genes_of_interest = {
                    # 'ENSG00000196776.17':[' CD47','left'],
                    # 'ENSG00000100030.15':['MAPK1 ','right'],
                     # 'ENSG00000173848.19':[' NET1','right'],
                     # 'ENSG00000100697.17':['DICER1','right'],
                    'ENSG00000001167.15':['NFYA','right'],
                     # 'ENSG00000100316.16':['RPL3','left'],
                     # 'ENSG00000174444.15':['RPL4','right']
                    }

tmp = relative_usages.loc[relative_usages['gene'].isin(genes_of_interest)]
tmp[samples_list] = np.round(tmp[samples_list],2)
tmp[index_cols+['dTotal-DMSO-1','dTotal-DMSO-2','dTotal-DMSO-3']]

In [ ]:
relative_usages = relative_usages.rename(columns={'gene':'gene_id'})
relative_usages = pd.merge(genes[['gene_id','gene_name','gene_type']],relative_usages,how='right',on='gene_id')

In [ ]:
out = subprocess.check_output('mkdir -p '+subdirs['tables_dir']+'APA/fractionation_and_HCT/', shell=True)
out_file_path = subdirs['tables_dir']+'APA/fractionation_and_HCT/relative_usages.tsv'
relative_usages.to_csv(out_file_path,sep=str('\t'),header=True,index=None,quoting=csv.QUOTE_NONE)

##### for NFYA project

In [ ]:
# run when relative_usages.tsv was collected (see below)
relative_usages_df = pd.read_csv(subdirs['tables_dir']+'APA/fractionation_and_HCT/relative_usages.tsv',delimiter="\t",index_col=None,header=0,)

sel_gene_name, sel_polyAsite_exon_idx = 'NFYA',3

PAS_labels = {1:'NFYA-A',2:'NFYA-B',3:'NFYA-C',4:'NFYA-D'}

temp_df = relative_usages.copy()
sel_PASusage_df = pd.DataFrame(temp_df.loc[(temp_df['gene_name']==sel_gene_name)&(temp_df['polyAsite_exon_idx']==sel_polyAsite_exon_idx)][order_sample_list].transpose())
sel_PASusage_df.columns = ['PAU']

sel_PASusage_df['sample'] = sel_PASusage_df.index
sel_PASusage_df = sel_PASusage_df.reset_index(drop=True)
sel_PASusage_df = pd.merge(sel_PASusage_df,sel_metadata,how='left',on='sample')
data = sel_PASusage_df.copy()

alpha_param, s_param = 0.9, 10
sns.set(font_scale=1.2)
sns.set_style("white")
fig, axes = plt.subplots(1, 1, sharey=False, sharex=False, figsize=(3.4,3.2))

y_feature,x_feature='condition','PAU'
order = ['whole cell', 'ER+mitochondria (1)','ER+mitochondria (2)','unclear (3)','nucleolus (4)','unclear (5)','membraneless cytosol (6)','membraneless cytosol (7)','membraneless cytosol (8)']
palette = ['mediumslateblue','#66CC00','tab:olive','lightgrey','deepskyblue','dimgrey','gold','chocolate','sienna']

ax = sns.stripplot(data = data,x=x_feature,y=y_feature,s=s_param,alpha=alpha_param,edgecolor='black',linewidth=0.5,
                     order = order, palette = palette,orient='h')
ax.tick_params(bottom=True,left=True)
ax.set(xlabel = PAS_labels[sel_polyAsite_exon_idx]+' PAS usage',ylabel='')

out = subprocess.check_output('mkdir -p '+subdirs['figures_dir']+'Fractionation_U2OS/over_genes/', shell=True)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+str(PAS_labels[sel_polyAsite_exon_idx])+'.relative_usage_in_TE.png',bbox_inches='tight',dpi=600)
fig.savefig(subdirs['figures_dir']+'Fractionation_U2OS/over_genes/'+str(PAS_labels[sel_polyAsite_exon_idx])+'.relative_usage_in_TE.pdf',bbox_inches='tight',dpi=600)